# Driver Drowsiness Detection – training on Kaggle

1. **Add data** → search `yawn-eye-dataset-new` (by serenaraju) and attach it.
2. Turn on **GPU** (Settings → Accelerator → GPU T4 x2 or P100).
3. Upload `train_model.py`, `train_dl_compare.py`, `train_classical.py` and `features.py` (Add data → Upload), then **Run All**.
4. Download `models_export.zip` from the Output tab and unzip it into your project's `models/` folder.

Order: main MobileNetV2 CNN → 5 CNN architectures compared → 11 classical ML models (they are compared against the CNNs).

In [ ]:
import os, glob, shutil
# Copy the uploaded scripts next to the notebook
for name in ['train_model.py', 'train_dl_compare.py', 'train_classical.py', 'features.py']:
    for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True):
        shutil.copy(p, f'/kaggle/working/{name}'); print('copied', p)
print(os.listdir('/kaggle/input'))

In [ ]:
!python /kaggle/working/train_model.py --task both --out /kaggle/working/models --epochs 10 --fine-tune-epochs 6

In [ ]:
# Deep learning comparison: Custom CNN, MobileNetV2, EfficientNetB0, ResNet50V2, DenseNet121
# (~30-60 min on a T4 GPU for both tasks; use --archs custom_cnn mobilenetv2 efficientnetb0 to save time)
%cd /kaggle/working
!python train_dl_compare.py --task both --out /kaggle/working/models --results /kaggle/working/results
print(open('/kaggle/working/results/eye_dl_comparison.md').read())

In [ ]:
# 11 classical ML models: LR, KNN, SVM, Decision Tree, Random Forest, Naive Bayes, Gradient Boosting,
# AdaBoost, Extra Trees, MLP (ANN), LDA (+ XGBoost if installed) – compared with the CNNs above
%cd /kaggle/working
!python train_classical.py --task both --out /kaggle/working/models --results /kaggle/working/results --tune
print(open('/kaggle/working/results/eye_classical_comparison.md').read())

In [ ]:
import json
from IPython.display import Image, display
for task in ['eye', 'yawn']:
    m = json.load(open(f'/kaggle/working/models/{task}_metrics.json'))
    print(task, {k: round(v, 4) for k, v in m['test_metrics'].items()})
    display(Image(f'/kaggle/working/models/{task}_training_curves.png'))
    display(Image(f'/kaggle/working/models/{task}_confusion_matrix.png'))

In [ ]:
# OPTIONAL – night / infrared eye model. Add data → search "MRL eye dataset" and attach it,
# and upload train_night.py. Skips itself if the dataset is not attached.
import glob, shutil
for p in glob.glob('/kaggle/input/**/train_night.py', recursive=True): shutil.copy(p, '/kaggle/working/train_night.py')
mrl = [d for d in glob.glob('/kaggle/input/*') if 'mrl' in d.lower()]
if mrl:
    %cd /kaggle/working
    !python train_night.py --data-dir "{mrl[0]}" --out /kaggle/working/models --results /kaggle/working/results --by-subject
else:
    print('MRL dataset not attached – skipping the night model')

In [ ]:
# Research rigour (upload significance.py, failure_gallery.py, gradcam.py too)
import glob, shutil
for name in ['significance.py', 'failure_gallery.py', 'gradcam.py']:
    for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True): shutil.copy(p, f'/kaggle/working/{name}')
%cd /kaggle/working
!python significance.py --task eye --models /kaggle/working/models --out /kaggle/working/results
!python failure_gallery.py --task eye --out /kaggle/working/results
print(open('/kaggle/working/results/eye_significance.md').read())

In [ ]:
# One zip to download from the Output tab
shutil.make_archive('/kaggle/working/models_export', 'zip', '/kaggle/working/models')